# Tạo Audio Kể Chuyện Kinh Thánh — GIỌNG NỮ MỚI (Nhúng Sẵn Giọng + Mode Preview + Thanh %)

1. Bật **Thời gian chạy → Đổi loại thời gian chạy → T4 GPU**.
2. Tích **`PREVIEW_MODE`** (mặc định `3` đoạn đầu ≈ 30–45 giây, render ~10 giây) để nghe thử giọng nữ mới ngay trên Colab; khi ưng thì bỏ tích để xuất nguyên tập.
3. Chọn `NFE_STEP = 16` (nhanh gấp đôi) hoặc `32` (tối đa).
4. Bấm **▶** → **chỉ cần chọn file kịch bản `*_script.md`** *(giọng nữ mẫu `voice_ref_nu_clean.wav` đã nhúng sẵn bên trong notebook)*.

In [ ]:
# @title CHỌN FILE KỊCH BẢN (.md / .txt) ĐỂ XUẤT MP3 TỰ ĐỘNG
SPEED = 1.05  # @param {type:"number"}
NFE_STEP = 16  # @param [16, 32] {type:"raw"}
PREVIEW_MODE = True  # @param {type:"boolean"}
PREVIEW_CHUNKS = 3  # @param {type:"slider", min:1, max:10, step:1}

import os
import gzip
import base64
import re
import sys
import shutil
import importlib.util
import subprocess
from pathlib import Path
from importlib.resources import files as pkg_files
from IPython.display import Audio, display
from google.colab import drive, files

# 1. Kết nối Google Drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

CACHE_DIR = Path('/content/drive/MyDrive/Bible_TTS_Cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
PIP_CACHE = CACHE_DIR / 'pip_cache'
PIP_CACHE.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/F5-TTS-Vietnamese')

# 2. Tải mã nguồn & cài nhanh (~15 giây, bỏ numpy<=1.26.4 gây biên dịch C 7 phút trên Python 3.13)
if not REPO_DIR.exists():
    print('>> Đang tải mã nguồn F5-TTS-Vietnamese...')
    subprocess.run(['git', 'clone', '-q', 'https://github.com/nguyenthienhy/F5-TTS-Vietnamese.git', str(REPO_DIR)], check=True)
    pyproject = REPO_DIR / 'pyproject.toml'
    txt = pyproject.read_text(encoding='utf-8')
    for drop in ['"numpy<=1.26.4",', '"gradio>=3.45.2",', '"transformers_stream_generator",', '"bitsandbytes>0.37.0; platform_machine != \'arm64\' and platform_system != \'Darwin\'",']:
        txt = txt.replace(drop, '')
    pyproject.write_text(txt, encoding='utf-8')

os.chdir(REPO_DIR)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

if importlib.util.find_spec('f5_tts') is None or shutil.which('f5-tts_infer-cli') is None:
    print('>> Đang cài gói F5-TTS (~15-20 giây)...')
    subprocess.run(['pip', 'install', '-q', '--cache-dir', str(PIP_CACHE), '-e', '.', 'omegaconf'], check=True)

import torch
import torchaudio
import soundfile as sf
from omegaconf import OmegaConf
from tqdm.auto import tqdm as tqdm_cls
from f5_tts.model import DiT, UNetT
from f5_tts.infer.utils_infer import (
    device as f5_device,
    load_vocoder,
    load_model,
    preprocess_ref_audio_text,
    chunk_text,
    infer_batch_process,
    target_rms,
    cross_fade_duration,
    cfg_strength,
    sway_sampling_coef,
)

if not torch.cuda.is_available():
    print('⚠️ CẢNH BÁO: Đang chạy CPU! Hãy bật Thời gian chạy -> Đổi loại -> T4 GPU.')
else:
    print(f'✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}')

# 3. Nạp Model & Giọng mẫu từ Google Drive Cache
model_dir = REPO_DIR / 'F5-TTS-Vietnamese-ViVoice'
model_dir.mkdir(exist_ok=True)
ckpt_local = model_dir / 'model_last.pt'
vocab_local = model_dir / 'vocab.txt'
ckpt_drive = CACHE_DIR / 'model_last.pt'
vocab_drive = CACHE_DIR / 'vocab.txt'

if not ckpt_local.exists() or not vocab_local.exists():
    if ckpt_drive.exists() and vocab_drive.exists():
        print('>> Đang nạp Model từ Google Drive Cache...')
        shutil.copy(ckpt_drive, ckpt_local)
        shutil.copy(vocab_drive, vocab_local)
    else:
        print('>> Chưa có trong Drive — đang tải Model từ Hugging Face (1 lần)...')
        from huggingface_hub import hf_hub_download
        for fname in ['config.json', 'model_last.pt']:
            hf_hub_download(repo_id='hynt/F5-TTS-Vietnamese-ViVoice', filename=fname, local_dir=str(model_dir))
        shutil.copy(model_dir / 'config.json', vocab_local)
        shutil.copy(ckpt_local, ckpt_drive)
        shutil.copy(vocab_local, vocab_drive)

# Giọng mẫu voice_ref_clean.wav (7.05s, 24kHz mono) đã nhúng sẵn trong notebook
_EMBEDDED_REF_WAV_GZ_B64 = '...'
ref_local = REPO_DIR / 'ref_voice_nu.wav'
ref_drive = CACHE_DIR / 'ref_voice_nu.wav'
_wav_data = gzip.decompress(base64.b64decode(_EMBEDDED_REF_WAV_GZ_B64))
ref_local.write_bytes(_wav_data)
ref_drive.write_bytes(_wav_data)
print('✅ Đã tự nạp giọng mẫu mới (voice_ref_nu_clean.wav nhúng sẵn, 8.05s)!')

REF_TEXT = 'chúng tôi cho rằng họ đã chế ngự được nó, buộc nó trở về đúng vị trí và nhắc cho nó nhớ về cái giá phải trả khi đi ngược lại với ý của họ.'

# Giữ sẵn Vocoder & Model trên RAM GPU suốt phiên Colab (từ lần thứ 2 không phải load lại)
cache = globals().get('_F5_CACHE')
if not cache or not Path(cache['ref_audio_proc']).exists() or cache.get('ref_text_raw') != REF_TEXT:
    print('>> Đang đưa Model & Vocoder lên GPU RAM (chỉ 1 lần trong phiên)...')
    vocoder = load_vocoder(vocoder_name='vocos', is_local=False, device=f5_device)
    model_cfg = OmegaConf.load(str(pkg_files('f5_tts').joinpath('configs/F5TTS_Base.yaml'))).model
    model_cls = {'DiT': DiT, 'UNetT': UNetT}[model_cfg.backbone]
    ema_model = load_model(
        model_cls,
        model_cfg.arch,
        str(ckpt_local),
        mel_spec_type='vocos',
        vocab_file=str(vocab_local),
        device=f5_device,
    )
    ref_audio_proc, ref_text_proc = preprocess_ref_audio_text(str(ref_local), REF_TEXT, device=f5_device)
    _F5_CACHE = {
        'vocoder': vocoder,
        'ema_model': ema_model,
        'ref_audio_proc': ref_audio_proc,
        'ref_text_proc': ref_text_proc,
        'ref_text_raw': REF_TEXT,
    }
    print('✅ Đã giữ sẵn Model trên GPU RAM!')
else:
    print('✅ Dùng lại Model đang nằm sẵn trên GPU RAM!')

vocoder = _F5_CACHE['vocoder']
ema_model = _F5_CACHE['ema_model']
ref_audio_proc = _F5_CACHE['ref_audio_proc']
ref_text_proc = _F5_CACHE['ref_text_proc']

def clean_script_text(raw: str) -> str:
    part_a = raw.split('## PHẦN B', 1)[0]
    kept = []
    for ln in part_a.splitlines():
        s = ln.strip()
        if not s or s.startswith('#') or set(s) <= {'-', '_'}: continue
        s = s.replace('**', '').replace('*', '').replace('`', '')
        kept.append(s)
    text = ' '.join(kept)
    text = re.sub(r'\[[^\]]+\]\s*', '', text)
    text = text.replace('—', ',').replace('"', '').replace('“', '').replace('”', '')
    text = re.sub(r'\s+', ' ', text).strip().lower()
    # Chuẩn hóa phát âm tên Kinh Thánh Công giáo & bỏ gạch nối cho F5-TTS đọc mượt
    tts_map = {
        'giê-su': 'giê xu',
        'giu-se': 'giu xe',
        'mô-sê': 'mô xê',
        'giô-suê': 'giô xuê',
        'giê-ru-sa-lem': 'giê ru xa lem',
        'sa-lô-môn': 'xa lô môn',
        'sa-un': 'xa un',
        'sa-mu-en': 'xa mu en',
        'bát-se-va': 'bát xe va',
        'bơ-e se-va': 'bơ e xe va',
        'sơ-va': 'xơ va',
        'cê-ci-li-a': 'xê xi li a',
    }
    for k, v in tts_map.items():
        text = text.replace(k, v)
    text = re.sub(r'(?<=[a-zà-ỹđ])-(?=[a-zà-ỹđ])', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

class _ColabProgress:
    @staticmethod
    def tqdm(iterable, **kwargs):
        return tqdm_cls(iterable, unit='đoạn', **kwargs)

# 4. Chọn 1 hoặc nhiều file kịch bản (.md / .txt) từ máy tính
print('>> Bấm Choose Files bên dưới để chọn file kịch bản (.md hoặc .txt)...')
uploaded_scripts = files.upload()

ref_tensor, ref_sr = torchaudio.load(ref_audio_proc)
ref_sec = ref_tensor.shape[-1] / ref_sr
max_chars = max(80, int(len(ref_text_proc.encode('utf-8')) / ref_sec * (22 - ref_sec)))

for fname in uploaded_scripts.keys():
    src_path = Path(fname)
    clean_txt = clean_script_text(src_path.read_text(encoding='utf-8'))
    src_path.unlink(missing_ok=True)
    if not clean_txt:
        print(f'⚠️ Bỏ qua {fname} vì không có nội dung sau khi làm sạch.')
        continue

    stem = Path(fname).stem.replace('_script_clean_voice', '').replace('_script', '')
    gen_batches = chunk_text(clean_txt, max_chars=max_chars)
    total_batches = len(gen_batches)
    if PREVIEW_MODE:
        gen_batches = gen_batches[:int(PREVIEW_CHUNKS)]
        out_mp3 = f'{stem}_preview.mp3'
        preview_words = sum(len(b.split()) for b in gen_batches)
        print(f'\n>> [PREVIEW MODE] Đang đọc {len(gen_batches)}/{total_batches} đoạn đầu ({preview_words} từ) của {fname} | NFE_STEP={int(NFE_STEP)}')
    else:
        out_mp3 = f'{stem}.mp3'
        word_count = len(clean_txt.split())
        est_audio_min = word_count / (200.0 * float(SPEED))
        print(f'\n>> Đang đọc FULL {fname}: {word_count} từ | {total_batches} đoạn | NFE_STEP={int(NFE_STEP)}')
        print(f'   ⏱️ Thời lượng audio ước tính: ~{est_audio_min:.1f} phút (xem thanh % và ETA bên dưới)')

    final_wave, final_sr, _ = next(
        infer_batch_process(
            (ref_tensor, ref_sr),
            ref_text_proc,
            gen_batches,
            ema_model,
            vocoder,
            mel_spec_type='vocos',
            progress=_ColabProgress,
            target_rms=target_rms,
            cross_fade_duration=cross_fade_duration,
            nfe_step=int(NFE_STEP),
            cfg_strength=cfg_strength,
            sway_sampling_coef=sway_sampling_coef,
            speed=float(SPEED),
            fix_duration=None,
            device=f5_device,
        )
    )

    out_audio_dir = REPO_DIR / 'out_audio'
    shutil.rmtree(out_audio_dir, ignore_errors=True)
    out_audio_dir.mkdir(exist_ok=True)
    raw_wav = out_audio_dir / f'{stem}.wav'
    sf.write(str(raw_wav), final_wave, final_sr)

    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'warning', '-i', str(raw_wav),
        '-af', 'loudnorm=I=-16:TP=-1.5:LRA=11',
        '-ar', '44100', '-ac', '2', '-b:a', '192k', out_mp3
    ], check=True)

    shutil.copy(out_mp3, CACHE_DIR / out_mp3)
    if PREVIEW_MODE:
        print(f'🎧 Nghe thử {out_mp3} ngay bên dưới (bỏ tích PREVIEW_MODE để xuất bản full):')
        display(Audio(out_mp3))
    else:
        print(f'✅ Hoàn tất {out_mp3}! Đã lưu Drive và đang tải về máy...')
        files.download(out_mp3)